# Auto Encoders for Image Compression


# Image Compression Steps:


Step-1: Imports

Step-2: Load Dataset

Step-3: Prepare Data

Step-4: Build AutoEncoder Model

Step-5: Train AutoEncoder

Step-6: Save AutoEncoder

Step-7: Load AutoEncoder Model

Step-8: Decode Function

Step-9: Visualize Results

Step-10: Compare byte sizes

## Imports

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers, losses
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.models import Model

In [2]:
import os
import numpy as np
import cv2

## Load Dataset

In [3]:
def load_data(img_dir):
  x_train = []
  #image_paths = [os.path.join(img_dir,w) for w in os.listdir(img_dir) if w.endswith(".tif")]
  image_paths = [os.path.join(img_dir,w) for w in os.listdir(img_dir) if w.endswith(".jpg")]
  image_paths = image_paths[:10000]
  for image_path in image_paths:
    img = cv2.imread(image_path, 0)
    img = cv2.resize(img, (752, 752))
    x_train.append(img)
  return np.array(x_train)

In [ ]:
data = load_data("drone imagery dataset")

# Image Compression

In [ ]:
data_raw = data

In [ ]:
data_raw.shape

## Prepare Data

In [ ]:
# test train split
split_rat = int(0.9*len(data_raw))
data = data_raw.astype('float32') / 255.
x_train = data[:split_rat]
x_test = data[split_rat:]
x_train = x_train[..., tf.newaxis]
x_test = x_test[..., tf.newaxis]
print(type(x_train[0]))
print (x_train.shape)
print (x_test.shape)

## Create Convolutional AutoEncoder Model

In [3]:
from keras.layers import Dense
@tf.keras.utils.register_keras_serializable()
class ConvAutoEncoder(Model):
  def __init__(self, **kwargs):
    super(ConvAutoEncoder, self).__init__(**kwargs)
    self.encoder = tf.keras.Sequential([
      layers.Input(shape=(752, 752, 1)),
      layers.Conv2D(32, (3,3), activation='relu', padding='same', strides=2),
      layers.Conv2D(16, (3,3), activation='relu', padding='same', strides=2),
      ])

    self.decoder = tf.keras.Sequential([
      layers.Conv2DTranspose(16, kernel_size=3, strides=2, activation='relu', padding='same'),
      layers.Conv2DTranspose(32, kernel_size=3, strides=2, activation='relu', padding='same'),
      layers.Conv2D(1, kernel_size=(3,3), activation='sigmoid', padding='same')])

  def call(self, x):
    encoded = self.encoder(x)
    decoded = self.decoder(encoded)
    return decoded

    @classmethod
    def from_config(cls, config):
        return cls(**config)

autoencoder = ConvAutoEncoder()

In [ ]:
autoencoder.compile(optimizer='adam', loss=losses.MeanSquaredError())

In [ ]:
autoencoder.encoder.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 376, 376, 32)   │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 188, 188, 16)   │         4,624 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,944 (19.31 KB)

 Trainable params: 4,944 (19.31 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
autoencoder.decoder.summary()

## Train AuoEncoder Model

In [ ]:
history = autoencoder.fit(x_train, x_train,
                epochs=20,
                shuffle=True,
                validation_data=(x_test, x_test),
                batch_size=1)

Epoch 1/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 194ms/step - loss: 1.5383e-04 - val_loss: 1.3495e-04
Epoch 2/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 162ms/step - loss: 1.3495e-04 - val_loss: 1.2152e-04
Epoch 3/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 292ms/step - loss: 1.2152e-04 - val_loss: 1.3312e-04
Epoch 4/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 155ms/step - loss: 1.3312e-04 - val_loss: 1.3954e-04
Epoch 5/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 126ms/step - loss: 1.3954e-04 - val_loss: 1.2896e-04
Epoch 6/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 121ms/step - loss: 1.2896e-04 - val_loss: 1.2536e-04
Epoch 7/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 137ms/step - loss: 1.2536e-04 - val_loss: 1.2936e-04
Epoch 8/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 98ms/step - loss: 1.2936e-04 - val_loss: 1.2698e-04
Epoch 9/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 98ms/step - loss: 1.2698e-04 - val_loss: 1.2550e-04
Epoch 10/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 96ms/step - loss: 1.2550e-04 - val_loss: 1.2834e-04
Epoch 11/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 135ms/step - loss: 1.2834e-04 - val_loss: 

## Test AutoEncoderModel

In [ ]:
encoded_imgs = autoencoder.encoder(x_test[:100]).numpy()
decoded_imgs = autoencoder.decoder(encoded_imgs).numpy()

## Visualize Results

In [ ]:
n = 10
skip=0
plt.figure(figsize=(20, 4))
for i in range(n):
  # display original
  ax = plt.subplot(2, n, i + 1)
  plt.imshow(tf.squeeze(x_test[skip+i]))
  plt.title("original")
  plt.gray()
  ax.get_xaxis().set_visible(False)
  ax.get_yaxis().set_visible(False)

  # display reconstruction
  ax = plt.subplot(2, n, i + 1 + n)
  plt.imshow(tf.squeeze(decoded_imgs[skip+i]))
  plt.title("reconstructed")
  plt.gray()
  ax.get_xaxis().set_visible(False)
  ax.get_yaxis().set_visible(False)
plt.show()

# Save and Load AutoEncoder Model

In [ ]:
autoencoder.save("autoencoder_model_updated_0_0001_10001img_003_drone.keras")

# Load Pretrained Model and Run Inference

In [4]:
autoencoder = tf.keras.models.load_model("autoencoder_model_updated_0_0001_10001img_003_drone.keras")

autoencoder.encoder.summary()
autoencoder.decoder.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)                    │ (None, 376, 376, 32)        │             320 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_4 (Conv2D)                    │ (None, 188, 188, 16)        │           4,624 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 4,944 (19.31 KB)

 Trainable params: 4,944 (19.31 KB)

 Non-trainable params: 0 (0.00 B)

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d_transpose_2 (Conv2DTranspose) │ (1, 376, 376, 16)           │           2,320 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_transpose_3 (Conv2DTranspose) │ (1, 752, 752, 32)           │           4,640 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_5 (Conv2D)                    │ (1, 752, 752, 1)            │             289 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 7,249 (28.32 KB)

 Trainable params: 7,249 (28.32 KB)

 Non-trainable params: 0 (0.00 B)

In [5]:
def run_inference(img):
  img = np.array([cv2.resize(img, (752,752))])
  img = img.astype('float32') / 255.
  img = img[..., tf.newaxis]
  encoded_imgs = autoencoder.encoder(img).numpy()
  decoded_imgs = autoencoder.decoder(encoded_imgs).numpy()
  return encoded_imgs, decoded_imgs

### Code to test the trained model for a single image

In [ ]:
im = cv2.imread("0000168_03306_d_0000174.jpg",1)

im = im[:,:,0]

# convert image to grayscale
plt.imshow(im, cmap='gray')

In [ ]:
embed, img = run_inference(im)

In [ ]:
plt.imshow(im, cmap='gray')

In [ ]:
plt.imshow(img[0], cmap = 'gray')

In [ ]:
plt.figure(figsize=(100, 4))
n=1
for i in range(n):
  # display original
  ax = plt.subplot(2, n, i + 1)
  plt.imshow(im)
  plt.title("original")
  plt.gray()
  ax.get_xaxis().set_visible(False)
  ax.get_yaxis().set_visible(False)

  # display reconstruction
  ax = plt.subplot(2, n, i + 1 + n)
  plt.imshow(tf.squeeze(img))
  plt.title("reconstructed")
  plt.gray()
  ax.get_xaxis().set_visible(False)
  ax.get_yaxis().set_visible(False)
plt.show()

# SECTION -- Finding the difference between original and decoded images --

## Code to convert source image to standard form

In [ ]:
from PIL import Image as im2
src= im
src
array = np.reshape(src, (752, 752))
print(array.shape)
data = im2.fromarray(array)
if data.mode != 'RGB':
    data = data.convert('RGB')
data.save('image 1 original.png')

## Code to convert decoded image to standard form

In [ ]:
from PIL import Image as im
output = img[0]
output = output * 255
output
array = np.reshape(output, (752, 752))
print(array.shape)
data = im.fromarray(array)
if data.mode != 'RGB':
    data = data.convert('RGB')
data.save('image 1 decoded.jpg')

## Code to find the difference between original image and decoded image

In [ ]:
from skimage.metrics import structural_similarity as ssim
import cv2
img1 = cv2.imread('image 1 original.png', 0)
img2 = cv2.imread('image 1 decoded.jpg', 0)
score = ssim(img1, img2, full=False)
print(f"SSIM: {score}")  

## Code to write encoded image as an array to output

In [ ]:
# save as bitmap image
from PIL import Image
endodedImageIntArr2 = np.array(endodedImageIntRounded)
endodedImageIntArr2=np.reshape(endodedImageIntArr2, (752,752))
endodedImageIntArr2 = endodedImageIntArr2.astype(np.uint8)
print(endodedImageIntArr2.shape)
data = Image.fromarray(endodedImageIntArr2)

if data.mode != 'RGB':
    data = data.convert('RGB')
data.save('image 1 decoded values.bmp')

# SECTION -- Code to find correlation between the encoded output layers --

In [ ]:
encoded_imgs = embed

In [ ]:
encoded_imgs.shape

In [ ]:
encoded_img = embed
encoded_img_orig = encoded_img

#convert value to truncated to 2 decimal places
encoded_img = encoded_img.reshape(565504)
encoded_img_truc = []
for item in encoded_img:
  item = round(item,2)
  encoded_img_truc.append(item)
print(len(encoded_img_truc))
encoded_img_truc=np.array(encoded_img_truc)
encoded_img_truc = encoded_img_truc.reshape(188,188,16)
#convert to a table of data with each column corresponding to a value
for item in encoded_img_truc:
  print(item)

In [ ]:
# convert the encoded data to table format
encoded_img_table = []
count =0
print(encoded_img_truc.shape)
for item in encoded_img_truc:
  #print(item.shape)
  for item2 in item:
    #print(item2.shape)
    if count<10:
      print(item2)
      pass
    encoded_img_table.append(item2)
    count+=1
print(len(encoded_img_table))

In [ ]:
#convert table to dataframe
import pandas as pd
df = pd.DataFrame(encoded_img_table)
df

In [ ]:
# find correleation between the columns of the df
df.corr()
# display in seaborn heatmap
import seaborn as sns
sns.heatmap(df.corr())

In [ ]:
# find correleation between the columns of the df
df.corr()
# display in seaborn heatmap
import seaborn as sns
plt.figure(figsize=(20, 20))
sns.heatmap(df.corr(), annot=True, fmt=".2f")
plt.show()

## find difference between 0 and 15 column

In [ ]:
# find difference between 0 and 15

diff0_15=df.iloc[:,15] - df2.iloc[:,15]
#diff0_15
diff0_15Arr = np.array(diff0_15)
#diff0_15Arr
# truncate to 2 decimal places
diff0_15Arr_Truc = []
sum=0
count =0
for item in diff0_15Arr:
  item = round(item,2)
  diff0_15Arr_Truc.append(item)
  sum += item
  count +=1
diff0_15Arr_Truc
print("average: " , (sum/count))


## convert the differnce between columns 0 and 15 to a int format

In [ ]:
#convert the differnce to a int format
diff_0_15_int =[]
sign_diff_0_15 = []
for i in diff0_15Arr_Truc:
  i = int(i*100)
  if i < 0:
    sign_diff_0_15.append('-')
    i = i * -1
  else:
    sign_diff_0_15.append('+')
  diff_0_15_int.append(i)
print (diff_0_15_int)
print (sign_diff_0_15)


# converting int to bytes with length of the array

In [ ]:
diff0_15Arr_Truc_bytes=[]
# converting int to bytes with length
# of the array as 2 and byter order as big
for i in diff_0_15_int:
  bytes_val = i.to_bytes(1, 'big')
  diff0_15Arr_Truc_bytes.append(bytes_val)

diff0_15Arr_Truc_bytes

# SECTION -- Code to perform linear regression on the different columns

In [ ]:
X=df.iloc[:,15]
y1= df.iloc[:, 0]

X_all=[]
y1_all=[]

X_all.append(X)
y1_all.append(y1)

X=df.iloc[:,15]
y1= df.iloc[:, 2]

X_all.append(X)
y1_all.append(y1)

X=df.iloc[:,15]
y1= df.iloc[:, 11]

X_all.append(X)
y1_all.append(y1)

X=df.iloc[:,15]
y1= df.iloc[:, 7]

X_all.append(X)
y1_all.append(y1)

X=df.iloc[:,15]
y1= df.iloc[:, 9]

X_all.append(X)
y1_all.append(y1)

X=df.iloc[:,15]
y1= df.iloc[:, 11]

X_all.append(X)
y1_all.append(y1)

X=df.iloc[:,15]
y1= df.iloc[:, 13]

X_all.append(X)
y1_all.append(y1)


reg_all=[]
from sklearn.model_selection import train_test_split

for k,i in enumerate(X_all):
  X1=i
  y11 = y1_all[k]


  X1_train, X1_test, y1_train, y1_test = train_test_split(X1,y11 ,
                                    test_size=0.10,shuffle = False
                                    )
  X1_train = np.array(X1_train)
  y1_train = np.array(y1_train)
  X1_test = np.array(X1_test)
  y1_test = np.array(y1_test)

  from sklearn.linear_model import LinearRegression
  reg = LinearRegression().fit(X1_train[:, np.newaxis], y1_train[:, np.newaxis])
  #reg = LinearRegression().fit(X[:, np.newaxis], y1[:, np.newaxis])
  reg_all.append(reg)

  y1_result = reg.predict(X1_test[:, np.newaxis])
  from sklearn.metrics import mean_squared_error
  from sklearn.metrics import mean_absolute_error
  print(y1_test)
  print(y1_result)
  print(mean_squared_error(y1_test, y1_result))
  print(mean_absolute_error(y1_test,y1_result))

In [ ]:
singleVal = []
singleVal.append(1.2)
singleVal = np.array(singleVal)
reg.predict(singleVal[:, np.newaxis])

In [ ]:
# save model in pickle
import pickle

for k,i in enumerate(reg_all):
# save
  with open('regression_model_8_'+str(k)+'.pkl','wb') as f:
      pickle.dump(i,f)

## Save the first column (0th) as a binary file

In [ ]:
# write binary output to file
#select the first column of df
first_col=df.iloc[:,0]
first_col = np.array(first_col)
#endodedImageIntRounded_bytes_0thCol = endodedImageIntRounded_bytes[:4096]


# convert first col values to int
first_col_int = []
for item in first_col:
  item = int(item*100)
  first_col_int.append(item)

# convert first col int to binary
first_col_int_bytes = []
for i in first_col_int:
  bytes_val = i.to_bytes(1, 'big')
  first_col_int_bytes.append(bytes_val)

endodedImageIntRounded_bytes_0thCol = first_col_int_bytes

#endodedImageIntRounded_bytes_0thCol
with open("encoded_0th_column.bin", "wb") as binary_file:

    # Write bytes to file
    for byte_data in endodedImageIntRounded_bytes_0thCol:  # Iterate through the list and write each bytes object
        binary_file.write(byte_data)
print('Binary output written successfully')


Binary output written successfully


# SECTION - Start with the regression model output and reconstruct the table of encoded values and reconstruct image

In [ ]:
# load model in pickle
import pickle
with open('regression_model_0_15.pkl', 'rb') as f:
    reg = pickle.load(f)


## Load 0th column to memory from file

In [ ]:
# read binary output to file

endodedImage_bytes_0thCol_read = []
with open("encoded_0th_column.bin", "rb") as binary_file:
    # Read bytes to file
    data = binary_file.read()
    # Convert to bytearray if needed
    endodedImage_bytes_0thCol_read = bytearray(data)

print('Binary output read successfully')
endodedImage_bytes_0thCol_read[1]

## Convert encoded read 0th column to float format

In [ ]:
endodedImage_float_0thCol_read = []
for item in endodedImage_bytes_0thCol_read:
  item = item / 100
  endodedImage_float_0thCol_read.append(item)
endodedImage_float_0thCol_read[0]

## Construct the new 15th column

In [ ]:
cols_source= [15,15]
cols_destination = [0,2]
cols_predict_all=[]
for i in range(len(cols_source)):
  endodedImage_float_Col_read_array = np.array(df.iloc[:,cols_source[i]])
  reg = reg_all[i]
  col_predict = reg.predict(endodedImage_float_Col_read_array[:, np.newaxis])
  cols_predict_all.append(col_predict)


In [ ]:
for i in range(len(cols_source)):
  df2.iloc[:,cols_destination[i]] = cols_predict_all[i]
df2

In [ ]:
# write the predicted values to new df, overwrite the 15 column with new values
df2 = df.copy()
#df

In [ ]:
df2

In [ ]:
# reconstruct encoded table from df
encoded_image_table_reconst = []
df2arr = np.array(df2)
df2arr = df2arr.reshape(1, 64, 64, 16)
encoded_image_table_reconst.append(df2arr)

In [ ]:
decoded_imgs = autoencoder.decoder(encoded_image_table_reconst).numpy()

In [ ]:
from PIL import Image as im
output = decoded_imgs[0]
output = output * 255
output
array = np.reshape(output, (752, 752))
print(array.shape)
data = im.fromarray(array)
if data.mode != 'RGB':
    data = data.convert('RGB')
data.save('image 1 decoded reconstructed from correleation.png')

In [ ]:
import time
# Check if the image was loaded correctly

# OPTIONAL: resize the image to match the model's input requirements
im = cv2.imread("0000168_03306_d_0000174.jpg",0)
im_resized = cv2.resize(im, (752, 752))

# Record the start time
start_time = time.time()

# Run the function you want to time
embed, img = run_inference(im_resized)

# Record the end time
end_time = time.time()

# Calculate the total time taken
time_taken = end_time - start_time

print(f"The inference took {time_taken:.4f} seconds to complete.")
#plt.imshow(img, cmap='gray')
#plt.show()

In [ ]:
# read image to local variable using open cv
import cv2

image = cv2.imread("0000168_03306_d_0000174.jpg")

image2 = cv2.imread("image 1 decoded.bmp")

import cv2
from skimage.metrics import structural_similarity as ssim

imge = image[:, :, 0]
imge2 = image2[:, :, 0]
gray_image1 = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
#gray_image1 = image
gray_image2 = cv2.cvtColor(image2, cv2.COLOR_BGR2GRAY)
# extract the blue channel from the image2

print(image2.shape)
(score, diff) = ssim(gray_image1, gray_image2, full=True)

print("SSIM:", score)